# Ôn tập Buổi 01 - Data Science Foundation, Python và OOP

        **Thời lượng gợi ý:** 45-60 phút  
        **Cách học:** trả lời câu hỏi trước khi chạy cell; sau mỗi ví dụ, tự nói thành lời *đầu vào - phép biến đổi - đầu ra*.

        ## Mục tiêu

        - Đặt một thao tác vào đúng bước của Data Science workflow.
- Phân biệt class, object, attribute, method và object state.
- Hiểu vì sao API Data Science thường dùng `fit()`, `transform()` và `predict()`.

        > Notebook này là tài liệu ôn chủ động, không thay thế toàn bộ slide. Khi một câu tự kiểm tra chưa chắc, quay lại đúng mục tương ứng trong `slides/buoi1_python_datascience.pdf`.


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in (HERE, *HERE.parents) if (p / "datasets").exists() and (p / "slides").exists()),
    None,
)
assert ROOT is not None, "Hãy mở notebook từ bên trong repo Hoan-Data-Science-Course."
import sys
print(f"Python: {sys.version.split()[0]}")
print(f'Repo: {ROOT}')


## 0. Chẩn đoán nhanh - chưa chạy code

**1. Trong `df.head()`, đâu là object và đâu là method?**

<details><summary>Kiểm tra đáp án</summary>

`df` là object; `head` là method của object đó.

</details>

**2. Data Cleaning đứng trước hay sau EDA?**

<details><summary>Kiểm tra đáp án</summary>

Thường đứng trước EDA chi tiết, nhưng kiểm tra khám phá ban đầu có thể lặp lại để định hướng cleaning.

</details>

**3. Sau `fit()`, một scaler lưu điều gì?**

<details><summary>Kiểm tra đáp án</summary>

Object state đã học từ training data, ví dụ mean và standard deviation.

</details>


## 1. Bản đồ tổng thể

```text
Business question -> Collect -> Clean -> EDA -> Features -> Model -> Evaluate -> Communicate/Deploy
```

Công cụ chỉ có ý nghĩa khi gắn với câu hỏi. `dropna()` thuộc Cleaning; `hist()` thuộc EDA; `predict()` thuộc Modeling/Inference.


In [ ]:
workflow = {
    "dropna": "Cleaning",
    "hist": "EDA",
    "fit": "Modeling",
    "precision": "Evaluation",
}
assert workflow["dropna"] == "Cleaning"
workflow


## 2. Built-in object trước khi tự tạo class

**Đoán trước:** sau cell dưới, `unique_labels` có giữ thứ tự và phần tử trùng không?


In [ ]:
features = ["age", "income", "age"]
customer = {"id": "C001", "monthly_fee": 25.0}
unique_labels = set(features)

print(features[0], customer["id"], unique_labels)
assert unique_labels == {"age", "income"}


## 3. Class, object, attribute, method

`DatasetSummary` đóng gói dữ liệu và hành vi liên quan. Property `n_rows` được tính từ state hiện tại thay vì lưu lặp lại.


In [ ]:
class DatasetSummary:
    dataset_count = 0

    def __init__(self, name, values):
        self.name = name
        self.values = list(values)
        DatasetSummary.dataset_count += 1

    @property
    def n_rows(self):
        return len(self.values)

    def mean(self):
        if not self.values:
            raise ValueError("Không thể tính mean của dữ liệu rỗng")
        return sum(self.values) / self.n_rows

    def __repr__(self):
        return f"DatasetSummary(name={self.name!r}, n_rows={self.n_rows})"

scores = DatasetSummary("scores", [7.0, 8.5, 9.0])
print(scores)
print("mean =", scores.mean())
assert scores.n_rows == 3 and scores.mean() == 8.166666666666666


## 4. Object state và giao diện `fit/transform`

**Điểm chống data leakage:** chỉ `fit()` trên training data; validation/test chỉ được `transform()` bằng state đã học.


In [ ]:
class MeanCenterer:
    def fit(self, values):
        self.mean_ = sum(values) / len(values)
        return self

    def transform(self, values):
        if not hasattr(self, "mean_"):
            raise RuntimeError("Phải fit trước khi transform")
        return [value - self.mean_ for value in values]

    def fit_transform(self, values):
        return self.fit(values).transform(values)

train = [10, 12, 14]
test = [20, 22]
centerer = MeanCenterer().fit(train)
train_centered = centerer.transform(train)
test_centered = centerer.transform(test)

print(centerer.mean_, train_centered, test_centered)
assert centerer.mean_ == 12
assert test_centered == [8, 10]


## Bài tự luyện

    Viết class `MinMaxScalerLite` có `fit()` lưu `min_`, `max_` và `transform()` đưa dữ liệu về `[0, 1]`. Báo lỗi nếu `max_ == min_`.

    <details><summary>Gợi ý / đáp án tham khảo</summary>

    ```python
    class MinMaxScalerLite:
def fit(self, values):
    self.min_ = min(values)
    self.max_ = max(values)
    if self.max_ == self.min_:
        raise ValueError("Không thể scale khi mọi giá trị bằng nhau")
    return self

def transform(self, values):
    return [(x - self.min_) / (self.max_ - self.min_) for x in values]
    ```

    </details>


## Phiếu rời buổi

        Không nhìn lại notebook, hãy tự xác nhận:

        - [ ] Tôi giải thích được pipeline Data Science bằng một ví dụ thực tế.
- [ ] Tôi phân biệt được class/object và attribute/method.
- [ ] Tôi giải thích được object state và vì sao chỉ fit trên train.

        Nếu chưa đánh dấu được một mục, ghi lại **một ví dụ do chính bạn nghĩ ra** rồi chạy thử.
